# Recall Bench 2.0.3

Same 25 frozen questions, same answer key, same judges as v2. This version fixes measurement problems in v2.0.2.

**What changed and why**

| Change | Reason |
|---|---|
| `MAX_OUT` 4,000 → 12,000 | Reasoning can eat the output budget. A config with many empty answers measures truncation, not ability. |
| DeepSeek pinned to Baseten | A provider that answers every call with 429 loses whole configs; pin to one that serves the model. |
| Kimi/GLM stay on Baseten with explicit quant tags (`baseten/fp8`, `baseten/fp4`) | A bare `baseten` pin does not exist for Kimi; the resolver catches it. Quantization is part of the model. |
| Empty answers logged as their own field, abort guard checks a rolling window | A guard that only looks at the first six calls lets a provider that fails later keep failing for the rest of the config. |
| `finish` reason recorded and surfaced per config | `length` vs `stop` is how you tell truncation from a bad answer without reading 75 files. |
| Reruns are per-config, driven by a stale-config list | Everything already on disk from v2 is reused. Only broken configs re-spend. |

Reads and writes `./local-data/v2/`; the same runs directory as v2.0.2, so good runs carry over. Run cells in order; cell 1 will tell you exactly which configs it plans to redo before it spends anything.

**Bring your own data.** No data ships with this notebook. It reads the same three files as `recall-bench-v2.ipynb`, under `./local-data/`: `data/recalls_contestant.csv`, `data/answer_key.json` and `questions/questions.json` (see the note at the top of that notebook for their format). Cell 0 prints `MISSING` for any it cannot find.

The deterministic grader (the cell headed `3. deterministic grader`, notebook cell index 5) hard-codes the original
question ids q01 to q15: their units (`event` for q02, q09 and q12, `record` for the rest), the totals a count
must sum to (q04, q05, q06 and q11 to 100, q12 to 52), a named firm for q09 and a wider tolerance for q14.
`type` is `deterministic` (graded by that cell, and rep 1 also by the judges) or `interpretive` (judges only).
Rewrite that cell for your own questions.
The judge prompt (the cell headed `4. two-judge pass`, notebook cell index 6) says "a CSV of 100 FDA food
enforcement records", and the runner's `SYSTEM` and `RULES` text (the cell headed `2. grid runner`, notebook cell index
4) describe the original CSV and its reason categories; rewrite them too.
Notebook cell index 3 appends one hard-coded redo entry, `("gemini-3.8-flash", "high")`, to `STALE`; drop or
replace it for your own run.

In [ ]:
# ── 0. setup, endpoint resolver, effort probe, smoke test ───────────
import os, json, time, pathlib, requests
from openai import OpenAI
# --- Spend guard -----------------------------------------------------------
# Every paid call is reserved at its worst case before it is sent and settled after, under one lock, so reserved
# plus settled never passes the cap even with all workers in flight. A call is refused, and the run stops, if its
# model has no confirmed price, if it has no output cap, or if its worst case would take the run past RUN_BUDGET_USD.
# PRICE holds each OpenRouter model's confirmed price and PRICE_DIRECT each direct OpenAI model's, in USD per
# million tokens (input, output). A model with no confirmed price is left out and the notebook refuses to run.
# Environment: RUN_BUDGET_USD (cap per run, default 25.00), SPEND_LEDGER_DIR (ledger and lock root, default
# spend_ledger/ at the repo root, found by the nearest .git above the working directory; refuses if there is none).
import math
from datetime import datetime, timezone


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output), retrieved 2026-11-05 from
# https://openrouter.ai/api/v1/models/<slug>/endpoints (or the model page on openrouter.ai for the same slug).
# Each figure is the highest price across that model's OpenRouter endpoints, so reservations are worst case.
PRICE = {
    "anthropic/claude-fable-5.1": (10.0, 50.0),
    "anthropic/claude-opus-5": (10.0, 50.0),
    "anthropic/claude-sonnet-5": (2.2, 11.0),
    "google/gemini-3.8-flash": (1.35, 6.75),
    "x-ai/grok-4.6": (8.0, 24.0),
    "moonshotai/kimi-k3": (4.5, 22.5),
    "z-ai/glm-5.3": (2.8, 8.8),
    "deepseek/deepseek-v4-pro-0813": (1.98, 8.0),
    "openai/gpt-5.6-sol": (8.8, 33.0),
    "nvidia/nemotron-3-ultra-550b-a55b": (0.625, 3.125),
    "qwen/qwen3.8-max": (2.0, 6.0),
    "qwen/qwen3.8-flash": (0.15, 0.47),
    "meta/muse-spark-1.3": (1.25, 4.25),
    "minimax/minimax-m3": (0.6, 2.4),
    "google/gemini-3.1-pro-preview": (7.2, 32.4),
}
# Direct OpenAI call (the Astra judge), long-context standard rate, the highest for a plain request. Source:
# https://developers.openai.com/api/docs/pricing , retrieved 2026-11-05. No direct Bedrock model is used here and none
# is priced, so guarded_converse refuses every model id.
PRICE_DIRECT = {
    "gpt-6-astra": (20.0, 75.0),
}
# Every model this notebook can call. FIELD and JUDGES below must stay inside these lists.
REQUIRED_OPENROUTER = ["anthropic/claude-fable-5.1", "anthropic/claude-opus-5", "anthropic/claude-sonnet-5",
                       "google/gemini-3.8-flash", "x-ai/grok-4.6", "moonshotai/kimi-k3", "z-ai/glm-5.3",
                       "deepseek/deepseek-v4-pro-0813", "openai/gpt-5.6-sol", "nvidia/nemotron-3-ultra-550b-a55b",
                       "qwen/qwen3.8-max", "qwen/qwen3.8-flash", "meta/muse-spark-1.3", "minimax/minimax-m3",
                       "google/gemini-3.1-pro-preview"]
REQUIRED_DIRECT = ["gpt-6-astra"]
missing_prices = sorted(set(REQUIRED_OPENROUTER) - set(PRICE)) + sorted(set(REQUIRED_DIRECT) - set(PRICE_DIRECT))
if missing_prices:
    raise RuntimeError(f"Not run: no confirmed price for {missing_prices}. Look up the listed input and output price in "
                       "USD per million tokens and add it to PRICE (OpenRouter models) or PRICE_DIRECT (direct OpenAI "
                       "models) in this cell, then rerun.")
for _m, (_pin, _pout) in {**PRICE, **PRICE_DIRECT}.items():
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) and x > 0 for x in (_pin, _pout)):
        raise RuntimeError(f"Not run: the price for {_m} is not a positive finite number. Fix it in this cell, then rerun.")


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


# One ledger and lock root for every notebook: SPEND_LEDGER_DIR, else spend_ledger/ at the repo root (the nearest .git above
# the working directory), else refuse.
LEDGER_ROOT = _ledger_root()
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(LEDGER_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. An exclusive lock file at LEDGER_ROOT/runner.lock, shared by every
# notebook: a second runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a
# runner that died without exiting) is never cleared automatically: confirm no runner is alive, then delete
# LEDGER_ROOT/runner.lock by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner (any notebook) may be running, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


import json
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body, prices=None):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    prices = PRICE if prices is None else prices
    model = body["model"]
    if model not in prices:
        raise BudgetExceeded(f"not sent: no confirmed price for {model}. Add it to "
                             f"{'PRICE' if prices is PRICE else 'PRICE_DIRECT'} in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = prices[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage, prices=None):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation. A direct call (prices given) reports
    no cost, so complete token counts are priced at the listed rates."""
    if prices is not None:
        if isinstance(usage, dict) and body["model"] in prices:
            ti, to = usage.get("prompt_tokens"), usage.get("completion_tokens")
            if _is_count(ti) and _is_count(to) and ti + to > 0:
                pin, pout = prices[body["model"]]
                c = pin / 1e6 * ti + pout / 1e6 * to
                if 0 < c < float("inf"):
                    return c, "usage"
        return worst_case_cost(body, prices), "worst_case"
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose, prices=None):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body, prices), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage, prices=None):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage, prices)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def _sdk_text(kw):
    """The request text, as a bound on input size (every field except the model and the output cap)."""
    return json.dumps({k: v for k, v in kw.items() if k not in ("model", "modelId", "extra_body")}, default=str,
                      ensure_ascii=False)


def guarded_chat(client, purpose, **kw):
    """OpenRouter chat call through the SDK. provider.max_price and usage include go in extra_body; the attempt is
    reserved at its worst case first and settled to the cost OpenRouter reports."""
    body = openrouter_body({"model": kw["model"], "messages": kw["messages"], "max_tokens": kw.get("max_tokens")})
    eb = dict(kw.get("extra_body") or {})
    eb["provider"] = {**(eb.get("provider") or {}), "max_price": body["provider"]["max_price"]}
    eb["usage"] = {"include": True}
    kw = {**kw, "extra_body": eb}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.chat.completions.create(**kw)
        try:
            usage = r.usage.model_dump() if r.usage is not None else None
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def guarded_responses(client, purpose, **kw):
    """Direct OpenAI Responses call. No OpenRouter fields apply. The attempt is reserved at the listed output price
    times max_output_tokens plus the listed input price times a byte-count bound on the prompt, from the same ledger,
    and refused if it does not fit. It settles on the reported token counts when complete, else at the worst case.
    PRICE_DIRECT must list the model."""
    body = {"model": kw["model"], "messages": [{"role": "user", "content": _sdk_text(kw)}],
            "max_tokens": kw.get("max_output_tokens")}
    aid = ledger_reserve(body, purpose, PRICE_DIRECT)
    usage = None
    try:
        r = client.responses.create(**kw)
        try:
            u = getattr(r, "usage", None)
            usage = {"prompt_tokens": getattr(u, "input_tokens", None),
                     "completion_tokens": getattr(u, "output_tokens", None)} if u is not None else None
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage, PRICE_DIRECT)


def guarded_converse(client, purpose, **kw):
    """Direct Bedrock Converse call. No OpenRouter fields apply. Reserved like guarded_responses against PRICE_DIRECT,
    which lists no Bedrock model, so every model id is refused. inferenceConfig.maxTokens bounds the output."""
    mt = (kw.get("inferenceConfig") or {}).get("maxTokens")
    body = {"model": kw["modelId"], "messages": [{"role": "user", "content": _sdk_text(kw)}], "max_tokens": mt}
    aid = ledger_reserve(body, purpose, PRICE_DIRECT)
    usage = None
    try:
        r = client.converse(**kw)
        try:
            u = r.get("usage") or {}
            usage = {"prompt_tokens": u.get("inputTokens"), "completion_tokens": u.get("outputTokens")}
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage, PRICE_DIRECT)




VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"
for d in ["runs", "judged", "results"]: (V2/d).mkdir(parents=True, exist_ok=True)

# max_retries=0: the SDK must not resend on its own, since every send needs its own reservation.
oai = OpenAI(timeout=300, max_retries=0)
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=300, max_retries=0)

FIELD = [  # (slug, label, [endpoint tags in order] or None = OpenRouter credit)
    ("anthropic/claude-fable-5.1",          "fable-5.1",        ["amazon-bedrock"]),
    ("anthropic/claude-opus-5",             "opus-5",           ["amazon-bedrock"]),
    ("anthropic/claude-sonnet-5",           "sonnet-5",         ["amazon-bedrock/global"]),
    ("google/gemini-3.8-flash",             "gemini-3.8-flash", ["google-vertex/global"]),
    ("x-ai/grok-4.6",                       "grok-4.6",         ["amazon-bedrock/us-west-2"]),
    ("moonshotai/kimi-k3",                  "kimi-k3",          ["baseten/fp8"]),
    ("z-ai/glm-5.3",                        "glm-5.3",          ["baseten/fp4"]),
    ("deepseek/deepseek-v4-pro-0813",       "deepseek-v4-pro",  ["baseten", "baseten/fp8", "baseten/fp4"]),   # resolver picks the one that exists
    ("openai/gpt-5.6-sol",                  "sol-via-or",       ["openai"]),
    ("nvidia/nemotron-3-ultra-550b-a55b",   "nemotron-3-ultra", ["baseten/fp4"]),
    ("qwen/qwen3.8-max",                    "qwen3.8-max",      None),
    ("qwen/qwen3.8-flash",                  "qwen3.8-flash",    None),
    ("meta/muse-spark-1.3",                 "muse-spark-1.3",   None),
    ("minimax/minimax-m3",                  "minimax-m3",       ["minimax/fp8"]),
]
EFFORTS = ["medium", "high"]
REPS = 3
MAX_OUT = 12000          # v2 used 4000; reasoning can use up a 4000 cap before the answer
JUDGES = {"astra": ("oai", "gpt-6-astra"), "gemini": ("orx", "google/gemini-3.1-pro-preview")}

def pin(tags): return {"provider": {"order": tags, "allow_fallbacks": False}} if tags else {}

for k in ["OPENAI_API_KEY", "OPENROUTER_API_KEY"]:
    print(f"{k}: {'ok' if os.environ.get(k) else 'MISSING'}")

def endpoints(slug):
    r = requests.get(f"https://openrouter.ai/api/v1/models/{slug}/endpoints", timeout=30)
    r.raise_for_status(); return r.json()["data"]["endpoints"]

# resolver: drop tags that don't exist, keep the first that does, fail loudly if none
print("\nendpoint check:")
RESOLVED = {}
for slug, label, tags in FIELD:
    if tags is None:
        eps = {e["tag"]: e for e in endpoints(slug)}
        cheap = min(eps.values(), key=lambda e: float(e["pricing"]["prompt"]))
        print(f"  {label:18s} OR-credit  cheapest {cheap['tag']} ${float(cheap['pricing']['prompt'])*1e6:.2f}/{float(cheap['pricing']['completion'])*1e6:.2f}")
        RESOLVED[label] = None; continue
    eps = {e["tag"]: e for e in endpoints(slug)}
    good = [t for t in tags if t in eps]
    if not good:
        print(f"  {label:18s} NO PINNED TAG EXISTS — available: {sorted(eps)}"); RESOLVED[label] = None; continue
    t = good[0]; e = eps[t]
    RESOLVED[label] = [t]
    extra = f"  (dropped {[x for x in tags if x not in eps]})" if len(good) < len(tags) else ""
    print(f"  {label:18s} {t:24s} ${float(e['pricing']['prompt'])*1e6:.2f}/{float(e['pricing']['completion'])*1e6:.2f}  quant={e.get('quantization')}{extra}")
FIELD = [(s, l, RESOLVED[l]) for s, l, _ in FIELD]

print("\nsmoke:")
PROBE = "Reply with the single word: ready"
for slug, label, tags in FIELD:
    try:
        r = guarded_chat(orx, "call", model=slug, max_tokens=400, messages=[{"role":"user","content":PROBE}], extra_body={"reasoning":{"effort":"low"}, **pin(tags)})
        got = getattr(r, "provider", None) or (r.model_extra or {}).get("provider")
        print(f"  [{label:18s}] ok  pinned={(tags[0] if tags else 'OR-credit'):24s} got={got}")
    except Exception as e:
        print(f"  [{label:18s}] FAILED: {str(e)[:130]}")
for name, (client, mid) in JUDGES.items():
    try:
        if client == "oai":
            r = guarded_responses(oai, "call", model=mid, input=PROBE, max_output_tokens=400, service_tier="default"); print(f"  [judge {name}] ok {r.model}")
        else:
            r = guarded_chat(orx, "call", model=mid, max_tokens=400, messages=[{"role":"user","content":PROBE}]); print(f"  [judge {name}] ok {r.model}")
    except Exception as e:
        print(f"  [judge {name}] FAILED: {str(e)[:130]}")

for f in ["data/recalls_contestant.csv", "data/answer_key.json", "questions/questions.json"]:
    p = VOL/f; print(f"{f}: {p.stat().st_size if p.exists() else 'MISSING'} bytes")


In [ ]:
# ── 1. triage: which configs need a rerun, and why ──────────────────
# Nothing is deleted here. This only reports. Cell 2 does the work.
import json, pathlib, pandas as pd
V2 = pathlib.Path("./local-data/v2"); RUNS = V2/"runs"
EMPTY_THRESHOLD = 0.10      # a config above this is measuring truncation, not ability

rows = []
for p in RUNS.glob("*.json"):
    r = json.loads(p.read_text())
    rows.append({"label": r.get("label"), "effort": r.get("effort"), "qid": r.get("qid"), "rep": r.get("rep"),
                 "is_error": "error" in r,
                 "is_empty": (not r.get("error")) and len(str(r.get("answer","")).strip()) == 0,
                 "finish": r.get("finish"), "out_tok": r.get("out_tok"), "reason_tok": r.get("reason_tok"),
                 "provider": r.get("provider")})
df = pd.DataFrame(rows)
if df.empty:
    print("no runs on disk; cell 2 will run the full grid"); STALE = []
else:
    g = df.groupby(["label","effort"]).agg(
            n=("qid","size"), errors=("is_error","sum"), empty_rate=("is_empty","mean"),
            truncated=("finish", lambda s: (s=="length").mean()), out_tok=("out_tok","mean"),
            provider=("provider", lambda s: s.mode().iloc[0] if len(s.dropna()) else None)).round(3)
    g["reason"] = ""
    g.loc[g["errors"] > 0, "reason"] += "errors;"
    g.loc[g["empty_rate"] > EMPTY_THRESHOLD, "reason"] += "empty>10%;"
    g.loc[g["n"] < 75, "reason"] += "incomplete;"
    print(g.sort_values(["reason","empty_rate"], ascending=[False,False]).to_string())
    STALE = [tuple(x) for x in g[g["reason"] != ""].index.tolist()]
    print(f"\n{len(STALE)} configs to redo at MAX_OUT={MAX_OUT}:")
    for lbl, eff in STALE: print(f"  {lbl} @ {eff}")
    print(f"\n{len(g)-len(STALE)} configs kept as-is.")

In [ ]:
STALE.append(("gemini-3.8-flash", "high"))
print(len(STALE), "configs to redo")

In [ ]:
# ── 2. grid runner (resumable; clears only stale configs) ───────────
import os, json, pathlib, time, random
from openai import OpenAI
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=300, max_retries=0)

VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"; RUNS = V2/"runs"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = qdoc["questions"]; BUDGET = qdoc["word_budget"]
CSV = (VOL/"data"/"recalls_contestant.csv").read_text()
ONLY = None          # e.g. ["deepseek-v4-pro"] to run one label
TAGS = {l: t for _, l, t in FIELD}

RULES = ("Reason categories, when asked: categorize on reason_for_recall only, case-insensitive, "
 "first match wins in this order: listeria='listeria'; salmonella='salmonella'; "
 "ecoli='e. coli'/'e.coli'/'stec'; other_micro=hepatitis/norovirus/clostridium/cronobacter/mold/yeast/botul; "
 "undeclared_allergen='undeclared'/'allergen'; foreign_material=foreign/metal/plastic/glass/wood/rubber; else other.")
SYSTEM = f"""You are analyzing a CSV of FDA food enforcement records. Answer only from the CSV. No outside knowledge, no tools.
Answer in at most {BUDGET} words. Anything past {BUDGET} words is discarded before grading. State the unit you used (records or events) whenever counts are involved.
Write your answer as plain text. Do not spend your output budget on internal reasoning; the answer itself is what is graded.
{RULES}"""

def pin(tags): return {"provider": {"order": tags, "allow_fallbacks": False}} if tags else {}
def run_id(label, effort, qid, rep): return f"{label}__{effort}__{qid}__r{rep}"

# clear stale configs (from cell 1) so they regenerate at the new cap
cleared = 0
for lbl, eff in (STALE or []):
    for p in RUNS.glob(f"{lbl}__{eff}__*.json"): p.unlink(); cleared += 1
print(f"cleared {cleared} run files across {len(STALE or [])} stale configs\n")

def call(slug, label, effort, q, rep):
    user = f"<csv>\n{CSV}\n</csv>\n\n<question id=\"{q['id']}\">\n{q['question']}\n</question>"
    err = "unknown"
    for attempt in range(6):
        t0 = time.time()
        try:
            r = guarded_chat(orx, "call", model=slug, max_tokens=MAX_OUT, messages=[{"role":"system","content":SYSTEM},{"role":"user","content":user}], extra_body={"reasoning":{"effort":effort}, "usage":{"include":True}, **pin(TAGS[label])})
            u = r.usage; ud = getattr(u, "model_extra", {}) or {}
            ptd = getattr(u, "prompt_tokens_details", None); ctd = getattr(u, "completion_tokens_details", None)
            ans = r.choices[0].message.content or ""
            return {"run_id": run_id(label,effort,q["id"],rep), "model_req": slug, "label": label,
                "model_resolved": r.model, "pinned": (TAGS[label] or [None])[0],
                "provider": ud.get("provider") or getattr(r,"provider",None) or (r.model_extra or {}).get("provider"),
                "effort": effort, "qid": q["id"], "rep": rep, "response_id": r.id,
                "answer": ans, "empty": len(ans.strip()) == 0, "finish": r.choices[0].finish_reason,
                "in_tok": u.prompt_tokens, "out_tok": u.completion_tokens,
                "cached_tok": getattr(ptd, "cached_tokens", None) if ptd else None,
                "reason_tok": getattr(ctd, "reasoning_tokens", None) if ctd else None,
                "or_cost": ud.get("cost"), "wall_s": round(time.time()-t0,1), "retries": attempt,
                "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
        except Exception as e:
            err = str(e)[:300]
            if any(k in err for k in ["429","502","503","504","timeout","Timeout","overloaded"]):
                time.sleep(min(90, 2**attempt + random.random()))
            else: break
    return {"run_id": run_id(label,effort,q["id"],rep), "model_req": slug, "label": label, "effort": effort,
            "qid": q["id"], "rep": rep, "error": err, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}

done = {p.stem for p in RUNS.glob("*.json")}
for slug, label, tags in FIELD:
    if ONLY and label not in ONLY: continue
    if tags is None and label not in ("qwen3.8-max","qwen3.8-flash","muse-spark-1.3"):
        print(f"== {label}: SKIPPED, no valid endpoint tag (see cell 0)"); continue
    for effort in EFFORTS:
        todo = [(q,rep) for rep in range(1,REPS+1) for q in QS if run_id(label,effort,q["id"],rep) not in done]
        if not todo: continue
        print(f"\n== {label} @ {effort}: {len(todo)} calls")
        tin=tout=tr=tc=0; cost=0.0; errs=0; empties=0; recent=[]; t0=time.time()
        for i,(q,rep) in enumerate(todo,1):
            rec = call(slug, label, effort, q, rep)
            (RUNS/f"{rec['run_id']}.json").write_text(json.dumps(rec, indent=1))
            bad = "error" in rec or rec.get("empty")
            recent.append(bad); recent = recent[-10:]
            if "error" in rec:
                errs += 1; print(f"  ERROR {rec['run_id']}: {rec['error'][:90]}")
            else:
                if rec.get("empty"): empties += 1
                tin+=rec["in_tok"]; tout+=rec["out_tok"]; tr+=rec["reason_tok"] or 0
                tc+=rec["cached_tok"] or 0; cost+=rec["or_cost"] or 0
            # rolling abort: 8 of the last 10 calls bad, anywhere in the config
            if len(recent) == 10 and sum(recent) >= 8:
                print(f"  ABORTING {label}@{effort}: 8 of last 10 calls failed or empty. Fix the route and rerun."); break
            if i % 15 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)} in={tin:,} cached={tc:,} out={tout:,} reason={tr:,} "
                      f"empty={empties} err={errs} or_cost=${cost:.2f} {time.time()-t0:.0f}s")
print("\nruns on disk:", len(list(RUNS.glob("*.json"))))


In [ ]:
# ── 3. deterministic grader (q01–q15), no model calls ───────────────
import json, re, pathlib, pandas as pd
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = {q["id"]: q for q in qdoc["questions"]}; BUDGET = qdoc["word_budget"]
DET = {q["id"] for q in qdoc["questions"] if q["type"]=="deterministic"}

def truncate(t): return " ".join(str(t).split()[:BUDGET])
def norm(t): return " ".join(re.sub(r"[^a-z0-9 ]", " ", str(t).lower()).split())
def nums(t):
    t = re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\1\2\3", str(t))
    return [float(x.replace(",","")) for x in re.findall(r"\d[\d,]*\.?\d*", t)]
UNIT  = {qid: "event" if qid in ("q02","q09","q12") else "record" for qid in QS}
TOTAL = {"q04":100,"q05":100,"q06":100,"q11":100,"q12":52}
ALIAS = {"": ["blank","missing","empty","unknown","none","null","no state","nan"]}
def key_alias(k):
    if k in ALIAS: return ALIAS[k]
    toks = [t for t in norm(str(k)).split() if t not in ("llc","inc","co","ltd","the")]
    return [" ".join(toks[:2]) if len(toks) > 1 else toks[0]]
def value_near(text, key_, val, window=60):
    t = norm(text)
    for a in key_alias(key_):
        for m in re.finditer(re.escape(a), t):
            if any(abs(n-float(val)) < 0.5 for n in nums(t[m.end():m.end()+window])): return True
    return False
def grade(q, answer):
    a = truncate(answer); ref = q["reference"]; qid = q["id"]; crit = {}
    if isinstance(ref, (int, float)):
        crit["value"] = any(abs(n-ref) <= (1.0 if qid=="q14" else 0.5) for n in nums(a))
    elif isinstance(ref, list):
        crit["value"] = all(any(abs(n-v) < 0.5 for n in nums(a)) for v in ref)
    elif isinstance(ref, dict):
        if qid == "q09":
            t = norm(a); crit["value"] = "gellert" in t and value_near(a, "GELLERT GLOBAL GROUP", 2)
            crit["others_one"] = bool(re.search(r"\b(all|every|each|remaining|other|others)\b[^.]{0,60}\b(1|one)\b", t)) or bool(re.search(r"\b(1|one)\b[^.]{0,60}\b(all|every|each|remaining|other|others)\b", t))
        else:
            crit["value"] = all(value_near(a, k, v) for k, v in ref.items())
    crit["unit"] = UNIT[qid] in norm(a)
    if qid in TOTAL:
        c = nums(a); crit["sums"] = any(abs(sum(c[i:j])-TOTAL[qid]) < 0.5 for i in range(len(c)) for j in range(i+2, len(c)+1))
    crit["over_budget"] = len(str(answer).split()) > BUDGET
    crit["empty"] = len(str(answer).strip()) == 0
    return crit

rows = []
for p in sorted((V2/"runs").glob("*.json")):
    r = json.loads(p.read_text())
    if "error" in r or r["qid"] not in DET: continue
    c = grade(QS[r["qid"]], r["answer"]); sk = [k for k in c if k not in ("over_budget","empty")]
    rows.append({"run_id": r["run_id"], "label": r["label"], "effort": r["effort"], "qid": r["qid"], "rep": r["rep"],
                 **c, "score": sum(c[k] for k in sk)/len(sk), "finish": r.get("finish"),
                 "reason_tok": r.get("reason_tok"), "answer": truncate(r["answer"])[:300]})
df = pd.DataFrame(rows); df.to_csv(V2/"judged"/"deterministic.csv", index=False)
print(f"{len(df)} graded answers\n")
piv = df.pivot_table(index=["label","effort"], columns="qid", values="value", aggfunc="mean").round(2)
piv["MEAN"] = df.groupby(["label","effort"]).value.mean().round(3)
piv["EMPTY"] = df.groupby(["label","effort"]).empty.mean().round(3)
print(piv.sort_values("MEAN", ascending=False).to_string())
fails = df[(~df.value) & (~df.empty)][["run_id","answer"]]
print(f"\n{len(fails)} non-empty value failures; first 15 (read these — they catch answer-key errors):")
for _, f in fails.head(15).iterrows(): print(f"- {f.run_id}\n    {f.answer[:200]}")


In [ ]:
# ── 4. two-judge pass: Astra (OpenAI direct) + Gemini 3.1 Pro (OR) ──
import os, json, re, pathlib, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
oai = OpenAI(timeout=300, max_retries=0)
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=300, max_retries=0)
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"; JD = V2/"judged"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = {q["id"]: q for q in qdoc["questions"]}; BUDGET = qdoc["word_budget"]
KEY = json.loads((VOL/"data"/"answer_key.json").read_text())
JUDGES = {"astra": ("oai", "gpt-6-astra"), "gemini": ("orx", "google/gemini-3.1-pro-preview")}
JUDGE_DET_REP1 = True; WORKERS = 8
def truncate(t): return " ".join(str(t).split()[:BUDGET])
def pin(tags): return {"provider": {"order": tags, "allow_fallbacks": False}} if tags else {}

# judge files whose run no longer exists (stale config was cleared) must go too
runs = {p.stem: json.loads(p.read_text()) for p in (V2/"runs").glob("*.json")}
orphans = [p for p in JD.glob("*__*.json") if p.stem.rsplit("__",1)[0] not in runs]
errored = [p for p in JD.glob("*__*.json") if "error" in json.loads(p.read_text())]
for p in set(orphans+errored): p.unlink()
print(f"cleared {len(set(orphans+errored))} judge files ({len(orphans)} orphaned by reruns, {len(errored)} errored)")

KEYX = json.dumps({k: KEY[k] for k in ["n_records","n_events","by_classification","reason_cat_records","reason_cat_events",
        "top_firms_records","top_states_records","non_US","median_lag_days_init_to_report","max_lag_days"]})
def judge_prompt(q, answer):
    facts = q.get("judge_facts") or f"reference value: {json.dumps(q['reference'])}"
    crits = "\n".join(f"{i+1}. {c}" for i, c in enumerate(q["rubric"]))
    return f"""You are grading one answer to one question about a CSV of 100 FDA food enforcement records. Grade each rubric criterion PASS or FAIL. Nothing else counts: not length, not tone, not extra correct facts. A criterion passes only if the answer establishes it; silence is FAIL. If the answer states a wrong number for something the facts specify, the related criterion is FAIL.

<facts_for_grader>
{facts}
</facts_for_grader>

<answer_key_excerpt>
{KEYX}
</answer_key_excerpt>

<question>
{q["question"]}
</question>

<rubric>
{crits}
</rubric>

<answer>
{answer}
</answer>

Return JSON only, no prose, no fences:
{{"criteria":[{{"n":1,"pass":true|false,"why":"<10 words"}}, ...], "fabrication": true|false}}
"fabrication" is true only if the answer states a specific firm, number, or fact that contradicts the facts or the key."""

def call_judge(name, q, answer):
    client, mid = JUDGES[name]; err = "unknown"
    for attempt in range(4):
        try:
            if client == "oai":
                r = guarded_responses(oai, "call", model=mid, input=judge_prompt(q, answer), reasoning={"effort":"medium"}, max_output_tokens=6000, service_tier="default")
                raw, model, tok = r.output_text, r.model, r.usage.total_tokens
            else:
                r = guarded_chat(orx, "call", model=mid, temperature=0, max_tokens=6000, messages=[{"role":"user","content":judge_prompt(q, answer)}], extra_body={"reasoning":{"effort":"low"}, **pin(["google-vertex/global"])})
                raw, model, tok = r.choices[0].message.content, r.model, r.usage.total_tokens
            raw = re.sub(r"^```(?:json)?|```$", "", (raw or "").strip(), flags=re.M).strip()
            out = json.loads(raw)
            if len(out["criteria"]) != len(q["rubric"]): raise ValueError("criteria count mismatch")
            return {"judge": name, "judge_model": model, "passes": [bool(c["pass"]) for c in out["criteria"]],
                    "why": [c.get("why","") for c in out["criteria"]], "fabrication": out.get("fabrication"), "judge_tok": tok}
        except Exception as e:
            err = str(e)[:200]; time.sleep(2**attempt + random.random())
    return {"judge": name, "error": err}

def work(item):
    r, q, name, out = item
    res = call_judge(name, q, truncate(r["answer"]))
    res.update({"run_id": r["run_id"], "qid": r["qid"], "label": r["label"], "effort": r["effort"],
                "rep": r["rep"], "qtype": q["type"]})
    out.write_text(json.dumps(res, indent=1)); return res

todo = []
for stem, r in runs.items():
    if "error" in r: continue
    if len(str(r.get("answer","")).strip()) == 0: continue     # don't spend judge calls on empty answers
    q = QS[r["qid"]]
    if q["type"] == "interpretive" or (JUDGE_DET_REP1 and r["rep"] == 1):
        for name in JUDGES:
            out = JD/f"{stem}__{name}.json"
            if not out.exists(): todo.append((r, q, name, out))
random.shuffle(todo); print(f"{len(todo)} judge calls, {WORKERS} workers")

tok = fails = done = 0; t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    for f in as_completed([ex.submit(work, it) for it in todo]):
        res = f.result(); done += 1
        if "error" in res: fails += 1; print(f"  FAIL {res['run_id']} [{res['judge']}]: {res['error'][:100]}")
        tok += res.get("judge_tok", 0)
        if done % 50 == 0 or done == len(todo):
            print(f"  {done}/{len(todo)}  judge_tok={tok:,}  fails={fails}  {time.time()-t0:.0f}s")
print("judging complete")


In [ ]:
# ── 5. rollup → v2/results/summary_2_0_3.csv ────────────────────────
import json, pathlib, pandas as pd, numpy as np
V2 = pathlib.Path("./local-data/v2")
det = pd.read_csv(V2/"judged"/"deterministic.csv")
for c in ["value","unit","sums","others_one","over_budget","empty"]:
    if c in det: det[c] = det[c].astype(str).str.lower()=="true"
jrows = []
for p in (V2/"judged").glob("*__*.json"):
    j = json.loads(p.read_text())
    if "error" in j: continue
    jrows.append({"run_id": j["run_id"], "label": j["label"], "effort": j["effort"], "qid": j["qid"], "rep": j["rep"],
                  "qtype": j["qtype"], "judge": j["judge"], "pass_rate": np.mean(j["passes"]),
                  "fabrication": bool(j.get("fabrication")), "passes": tuple(j["passes"])})
jd = pd.DataFrame(jrows)
trows = []
for p in (V2/"runs").glob("*.json"):
    r = json.loads(p.read_text())
    if "error" in r: continue
    trows.append({"label": r["label"], "effort": r["effort"], "in": r["in_tok"], "out": r["out_tok"],
                  "reason": r.get("reason_tok") or 0, "cached": r.get("cached_tok") or 0,
                  "or_cost": r.get("or_cost") or 0, "provider": r.get("provider"),
                  "empty": len(str(r.get("answer","")).strip())==0, "length_finish": r.get("finish")=="length"})
tok = pd.DataFrame(trows)
g = ["label","effort"]; interp = jd[jd.qtype=="interpretive"]
summary = pd.DataFrame({
    "det_value":    det.groupby(g).value.mean(),
    "empty_rate":   tok.groupby(g).empty.mean(),
    "truncated":    tok.groupby(g).length_finish.mean(),
    "int_astra":    interp[interp.judge=="astra"].groupby(g).pass_rate.mean(),
    "int_gemini":   interp[interp.judge=="gemini"].groupby(g).pass_rate.mean(),
    "fabrication":  interp.groupby(g).fabrication.mean(),
    "reason_tok":   tok.groupby(g).reason.mean(),
    "cached_share": tok.groupby(g).cached.sum() / tok.groupby(g)["in"].sum(),
    "or_cost":      tok.groupby(g).or_cost.sum(),
    "provider":     tok.groupby(g).provider.agg(lambda s: s.mode().iloc[0] if len(s.dropna()) else None),
}).round(3)
summary["int_mean"] = summary[["int_astra","int_gemini"]].mean(axis=1).round(3)
summary["overall"]  = ((summary.det_value*15 + summary.int_mean*10)/25).round(3)
# a config with a high empty rate is unranked: it measured truncation
summary["RANKED"] = summary.empty_rate <= 0.10
summary = summary.sort_values(["RANKED","overall"], ascending=[False,False])
summary.to_csv(V2/"results"/"summary_2_0_3.csv")
print("== RANKED (empty_rate <= 10%)\n", summary[summary.RANKED].drop(columns="RANKED").to_string())
print("\n== UNRANKED — measured truncation, not ability\n", summary[~summary.RANKED].drop(columns="RANKED").to_string())

w = jd[jd.qtype=="interpretive"].pivot_table(index="run_id", columns="judge", values="passes", aggfunc="first").dropna()
if len(w):
    agree = np.mean([a==b for pa,pb in zip(w.astra, w.gemini) for a,b in zip(pa,pb)])
    print(f"\njudge-vs-judge criterion agreement: {agree:.3f} over {len(w)} answers")
m = jd[jd.qtype=="deterministic"].merge(det[det.rep==1][["run_id","value"]], on="run_id")
if len(m):
    m["j1"] = m.passes.apply(lambda t: t[0])
    for jn, gg in m.groupby("judge"):
        print(f"judge {jn} vs python: agree={np.mean(gg.j1==gg.value):.3f} (lenient={np.mean(gg.j1 & ~gg.value):.3f}, strict={np.mean(~gg.j1 & gg.value):.3f})")
print("\neffort curve (overall, ranked configs only):")
print(summary[summary.RANKED].reset_index().pivot(index="label", columns="effort", values="overall").to_string())
print("\nwritten:", V2/"results"/"summary_2_0_3.csv")
